# FinGPT Two-Agent Signal Pipeline (Colab)

## Cell 1 — GPU check & install

This cell verifies GPU availability and installs notebook dependencies for the full pipeline demo.

In [ ]:
# Check GPU
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install dependencies
!pip install -q transformers accelerate vllm alpaca-trade-api yfinance pydantic python-dotenv tqdm

## Cell 2 — Clone repo and set path

This cell clones the repository in Colab and sets the working directory so project imports resolve correctly.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/FinGPT_Project'
repo_url = 'https://github.com/juankim834/FinGPT_Project.git'

if os.path.exists(project_path):
    print(f"Found repo at: {project_path}, updating")
    %cd {project_path}
    !git pull
else:
    print("Repo is not exist, try to clone intially")
    %cd /content/drive/MyDrive
    !git clone {repo_url}
    %cd {project_path}

# Shared paths on Google Drive for demo artifacts.
DEMO_OUTPUT_DIR = os.path.join(project_path, "output")
ARTICLE_CACHE_DIR = os.path.join(project_path, "cache")
os.makedirs(DEMO_OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTICLE_CACHE_DIR, exist_ok=True)
print(f"Drive output dir: {DEMO_OUTPUT_DIR}")
print(f"Article cache dir: {ARTICLE_CACHE_DIR}")

## Cell 3 — Load secrets

This cell loads Alpaca credentials from Colab Secrets.

In [ ]:
import os
from google.colab import userdata

# Choose article provider: "finnhub" or "alpaca"
os.environ["NEWS_PROVIDER"] = "finnhub"

# Finnhub credential (required when NEWS_PROVIDER=finnhub)
os.environ["FINNHUB_API_KEY"] = userdata.get("FINNHUB_API_KEY")

# Alpaca credentials (optional unless NEWS_PROVIDER=alpaca)
os.environ["ALPACA_API_KEY"] = userdata.get("ALPACA_API_KEY")
os.environ["ALPACA_API_SECRET"] = userdata.get("ALPACA_API_SECRET")

# Resolve local merged model directory for vLLM loading.
model_path_candidates = [
    "/content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm",
    "/content/MyDrive/deepseek_fingpt_outputs/merged_for_vllm",
]
resolved_model_path = None
for candidate in model_path_candidates:
    if os.path.isfile(os.path.join(candidate, "config.json")):
        resolved_model_path = candidate
        break

if resolved_model_path is None:
    raise FileNotFoundError(
        "Could not find local model folder with config.json. Checked: "
        + ", ".join(model_path_candidates)
    )

os.environ["FINGPT_MODEL_PATH"] = resolved_model_path

# Enable shared single-LLM mode for Agent 1 + Agent 2
os.environ["SHARE_SINGLE_LLM_BETWEEN_AGENTS"] = "true"

print("Colab secrets loaded and single-LLM mode configured.")
print(f"NEWS_PROVIDER={os.environ['NEWS_PROVIDER']}")
print(f"FINGPT_MODEL_PATH={os.environ['FINGPT_MODEL_PATH']}")

## Cell 4 — Load one shared model on vLLM and check VRAM

This cell loads one shared vLLM engine using the `FINGPT_MODEL_PATH` model ID and reports GPU memory before/after engine initialization.

In [ ]:
import os
import torch
from vllm import LLM, SamplingParams


def used_vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1e9


def gpu_supports_bf16() -> bool:
    if not torch.cuda.is_available():
        return False
    major, _minor = torch.cuda.get_device_capability(0)
    # Ampere+ generally supports fast BF16; older cards (e.g., T4) should use FP16.
    return major >= 8


if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this vLLM demo.")

print(f"VRAM before vLLM load: {used_vram_gb():.2f} GB")

model_id = os.environ.get("FINGPT_MODEL_PATH", "deepseek-ai/DeepSeek-R1-Distill-Llama-8B")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
prefer_dtype = "bfloat16" if gpu_supports_bf16() else "float16"

print(f"GPU detected: {gpu_name} ({total_vram_gb:.1f} GB)")
print(f"Model path: {model_id}")
print(f"Preferred dtype: {prefer_dtype}")

# Try safer startup configs first for Colab stability.
startup_profiles = [
    {"dtype": prefer_dtype, "gpu_memory_utilization": 0.85, "enforce_eager": True},
    {"dtype": "float16", "gpu_memory_utilization": 0.80, "enforce_eager": True},
]

llm = None
last_error = None
for i, profile in enumerate(startup_profiles, start=1):
    try:
        print(f"Attempt {i}: {profile}")
        llm = LLM(
            model=model_id,
            trust_remote_code=True,
            disable_log_stats=True,
            **profile,
        )
        break
    except Exception as exc:
        last_error = exc
        print(f"Attempt {i} failed: {type(exc).__name__}: {exc}")

if llm is None:
    raise RuntimeError(
        "vLLM failed to initialize after fallback attempts. "
        "Common causes: insufficient VRAM, incompatible dtype, or incomplete model files."
    ) from last_error

# Inject shared vLLM engine for Agent 1 extractor.
from agent1.extractor import set_shared_vllm_engine
set_shared_vllm_engine(llm)

# Warm up once so memory usage reflects an initialized inference path.
_ = llm.generate(["Warmup."], SamplingParams(max_tokens=1, temperature=0.0))

used_after_load = used_vram_gb()
fits_device = used_after_load < total_vram_gb
fits_40gb = used_after_load <= 40.0

print(f"VRAM after vLLM load: {used_after_load:.2f} GB")
print(f"vLLM engine ready for model: {model_id}")
print(f"Fits current GPU capacity ({total_vram_gb:.1f} GB): {fits_device}")
print(f"Fits within 40 GB budget: {fits_40gb}")

## Cell 5 — Fetch articles

This cell fetches 10 recent articles for a fixed ticker set and shows a preview table.

In [ ]:
import hashlib
import json
import os
import pandas as pd
from ingestion.news_fetcher import fetch_recent_articles

TICKERS = ["AAPL", "NVDA", "TSLA"]
ARTICLE_LIMIT = 10

cache_root = globals().get("ARTICLE_CACHE_DIR", "cache")
os.makedirs(cache_root, exist_ok=True)

cache_key_source = json.dumps(
    {"tickers": sorted(TICKERS), "limit": ARTICLE_LIMIT},
    sort_keys=True,
)
cache_key = hashlib.md5(cache_key_source.encode("utf-8")).hexdigest()[:12]
cache_path = os.path.join(cache_root, f"articles_{cache_key}.json")

if os.path.exists(cache_path):
    with open(cache_path, "r", encoding="utf-8") as handle:
        articles = json.load(handle)
    print(f"Loaded {len(articles)} article(s) from cache: {cache_path}")
else:
    articles = fetch_recent_articles(TICKERS, limit=ARTICLE_LIMIT)
    with open(cache_path, "w", encoding="utf-8") as handle:
        json.dump(articles, handle, indent=2)
    print(f"Fetched {len(articles)} article(s) and cached to: {cache_path}")

preview_df = pd.DataFrame(articles)
preview_cols = [c for c in ["headline", "source", "created_at", "summary"] if c in preview_df.columns]
display(preview_df[preview_cols].head(10))

## Cell 6 — Run Agent 1

This cell runs fingerprint extraction on each fetched article and prints per-article status. It also pretty-prints the first successful fingerprint.

In [ ]:
import json
from tqdm.auto import tqdm
from agent1.extractor import extract_fingerprint

fingerprints = []
first_successful_fp = None

for article in tqdm(articles, desc="Agent 1 extraction"):
    article_text = f"{article.get('headline', '')} {article.get('summary', '')}".strip()
    fp = extract_fingerprint(article_text)
    status = "OK" if fp is not None else "SKIPPED"
    print(f"{status} | {article.get('headline', '')[:90]}")
    if fp is not None:
        row = {
            "article": article,
            "fingerprint": fp,
        }
        fingerprints.append(row)
        if first_successful_fp is None:
            first_successful_fp = fp

print(f"\nExtracted {len(fingerprints)} fingerprint(s) out of {len(articles)} articles")
if first_successful_fp is not None:
    print("\nFirst successful NewsFingerprint:")
    print(json.dumps(first_successful_fp.model_dump(), indent=2))

## Cell 7 — Run Agent 2

This cell runs signal generation from extracted fingerprints and prints per-item status. It also pretty-prints the first successful signal.

In [ ]:
from tqdm.auto import tqdm
from agent2.reasoner import generate_signal

signals = []
first_successful_signal = None

for row in tqdm(fingerprints, desc="Agent 2 reasoning"):
    fp = row["fingerprint"]
    signal = generate_signal(fp)
    status = "OK" if signal is not None else "SKIPPED"
    print(f"{status} | {fp.headline[:90]}")
    if signal is not None:
        enriched = {
            "article": row["article"],
            "fingerprint": fp,
            "signal": signal,
        }
        signals.append(enriched)
        if first_successful_signal is None:
            first_successful_signal = signal

print(f"\nGenerated {len(signals)} signal(s) out of {len(fingerprints)} fingerprints")
if first_successful_signal is not None:
    import json
    print("\nFirst successful TradingSignal:")
    print(json.dumps(first_successful_signal.model_dump(), indent=2))

## Cell 8 — Results DataFrame

This cell builds a dataframe of signal + sentiment fields for quick inspection.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

rows = []
for row in tqdm(signals, desc="Building results table"):
    fp = row["fingerprint"]
    sig = row["signal"]
    rows.append(
        {
            "ticker": sig.ticker,
            "direction": sig.direction,
            "strategy_tag": sig.strategy_tag,
            "sentiment_score": fp.sentiment_score,
            "sentiment_confidence": fp.sentiment_confidence,
            "signal_confidence": sig.confidence,
            "cot": sig.cot,
        }
    )

results_df = pd.DataFrame(rows)
if results_df.empty:
    print("No signals produced.")
else:
    display(results_df)

## Cell 9 — Evaluation (mini)

This cell runs self-consistency evaluation on the fetched articles with N=3 runs and prints a per-article consistency table.

In [ ]:
from evaluation.evaluator import evaluate_self_consistency

article_texts = [f"{a.get('headline', '')} {a.get('summary', '')}".strip() for a in articles]
mini_eval = evaluate_self_consistency(article_texts, n_runs=3)

print(f"Consistency rate: {mini_eval['consistency_rate']:.3f}")
consistency_df = pd.DataFrame(mini_eval["per_article"])
display(consistency_df)

## Cell 10 — Save outputs

This cell saves the signal records to JSON for later analysis.

In [ ]:
import json
import os
from datetime import datetime, timezone
from tqdm.auto import tqdm

output_dir = globals().get("DEMO_OUTPUT_DIR", "output")
os.makedirs(output_dir, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
outpath = os.path.join(output_dir, f"signals_demo_{timestamp}.json")

serializable_signals = []
for row in tqdm(signals, desc="Serializing outputs"):
    serializable_signals.append(
        {
            "article": row["article"],
            "fingerprint": row["fingerprint"].model_dump(),
            "signal": row["signal"].model_dump(),
        }
    )

with open(outpath, "w", encoding="utf-8") as handle:
    json.dump(serializable_signals, handle, indent=2)

print(f"Saved {len(serializable_signals)} records to {outpath}")